# 10 — Testing whether representation size explains the model ranking

The native representations differ greatly in length: AlphaEarth contains 64 values, while DINOv2 contains 768. A longer vector may preserve more useful information, but it also gives the downstream model a larger feature space. This notebook tests whether the main ranking persists when seven representative feature sets are each reduced to 64 principal components.

Principal component analysis (PCA) creates 64 weighted combinations that retain the largest patterns of variation in the training data. Missing-value treatment, standardisation and PCA are repeated within each training split so that held-out boroughs do not influence the transformation. The compressed features are then evaluated with the same borough-based Ridge procedure.

## Main findings

Compression reduces the absolute performance of the larger representations. Relative to their native dimensions, mean R² falls by about 0.0661 for PTAL DINOv2 and 0.0478 for EPC DINOv2. The all-representation model falls by about 0.0371 for PTAL and 0.0692 for EPC. The discarded dimensions therefore contain genuine predictive information.

The broad ordering nevertheless remains stable. PTAL has exactly the same rank order before and after compression. For EPC, only DINOv2 and DINOv2 plus SatCLIP exchange adjacent positions. The Spearman rank correlation—a statistic ranging from −1 for reversed rankings to +1 for identical rankings—is 1.000 for PTAL and 0.964 for EPC. The all-representation set remains first for both outcomes.

Unequal dimensionality affects the level of performance but does not explain the central ranking conclusions.

In [ ]:
# Connect Google Drive and load packages for fold-specific PCA and Ridge fitting.
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys, json, gc, time, hashlib, platform, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import sklearn

from joblib import Memory
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

FINAL_CODE_DIR = Path("/content/drive/MyDrive/GEOG0105/CODE/FINAL_PIPELINE")
if str(FINAL_CODE_DIR) in sys.path:
    sys.path.remove(str(FINAL_CODE_DIR))
sys.path.insert(0, str(FINAL_CODE_DIR))

import importlib
import config as _config
importlib.invalidate_caches()
_config = importlib.reload(_config)
globals().update({name: getattr(_config, name) for name in dir(_config) if not name.startswith("_")})

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 240)

required_paths = [
    FINAL_MODEL_TABLE_PATH, FEATURE_MANIFEST_JSON_PATH, RIDGE_OUTER_FOLDS_PATH,
    RIDGE_RUN_SPEC_PATH, RIDGE_CORE_AUDIT_PATH, RIDGE_CORE_RESULTS_PATH,
    RIDGE_CORE_PREDICTIONS_PATH, IMAGE_LOCATION_RUN_SPEC_PATH,
    IMAGE_LOCATION_AUDIT_PATH, IMAGE_LOCATION_RESULTS_PATH,
    IMAGE_LOCATION_PREDICTIONS_PATH,
]
for path in required_paths:
    print(path, path.exists())
    assert path.exists(), f"Missing prerequisite: {path}"

print("Python:", platform.python_version())
print("numpy:", np.__version__, "pandas:", pd.__version__, "sklearn:", sklearn.__version__)

## 1. Load the native-dimension results and model definitions

The common table, feature manifest, borough assignments and native results are read from the main benchmark and DINOv2–SatCLIP analysis. Stored identifiers ensure that compressed and native models refer to the same data and features.

In [ ]:
# Read the common data and native-dimension source results.
df = pd.read_parquet(FINAL_MODEL_TABLE_PATH)
with open(FEATURE_MANIFEST_JSON_PATH, "r") as f:
    manifest = json.load(f)
with open(RIDGE_RUN_SPEC_PATH, "r") as f:
    source_06_spec = json.load(f)
with open(RIDGE_CORE_AUDIT_PATH, "r") as f:
    source_06_audit = json.load(f)
with open(IMAGE_LOCATION_RUN_SPEC_PATH, "r") as f:
    source_09_spec = json.load(f)
with open(IMAGE_LOCATION_AUDIT_PATH, "r") as f:
    source_09_audit = json.load(f)

assert source_06_audit["integrity_gate_pass"] and source_06_audit["interpretation_gate_pass"]
assert source_09_audit["integrity_gate_pass"] and source_09_audit["interpretation_gate_pass"]

target_col = manifest["target_column"]
group_col = manifest["group_column"]
categorical_master = set(manifest["categorical_columns"])
feature_sets = manifest["feature_sets"]

assert len(df) == 26597 and df["sample_id"].is_unique
assert df[target_col].notna().all() and df[group_col].notna().all()
assert df[group_col].nunique() == 33
df = df.sort_values(["task", "sample_id"], kind="mergesort").reset_index(drop=True)
task_counts = df["task"].value_counts().to_dict()
assert task_counts == {"EPC": 20000, "PTAL": 6597}

model_key_hash = hashlib.sha256(
    pd.util.hash_pandas_object(df[["sample_id", "task", group_col, target_col]], index=False).values.tobytes()
).hexdigest()
manifest_hash = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode("utf-8")).hexdigest()
source_06_spec_sha256 = hashlib.sha256(json.dumps(source_06_spec, sort_keys=True).encode("utf-8")).hexdigest()
source_09_spec_sha256 = hashlib.sha256(json.dumps(source_09_spec, sort_keys=True).encode("utf-8")).hexdigest()

assert source_06_spec["model_key_sha256"] == source_09_spec["model_key_sha256"] == model_key_hash
assert source_06_spec["feature_manifest_sha256"] == source_09_spec["feature_manifest_sha256"] == manifest_hash
assert source_06_spec["outer_fold_assignment_sha256"] == source_09_spec["outer_fold_assignment_sha256"]
assert int(source_06_spec["outer_splits"]) == int(RIDGE_OUTER_SPLITS) == 5
assert int(source_06_spec["inner_splits"]) == int(RIDGE_INNER_SPLITS) == 3
assert [float(x) for x in source_06_spec["alpha_grid"]] == [float(x) for x in RIDGE_ALPHA_GRID]

print("Frozen source identity: PASS")
print("Model-key SHA256:", model_key_hash)
print("Manifest SHA256:", manifest_hash)

## 2. Select the seven representation sets

The comparison includes SatCLIP, TESSERA, AlphaEarth, DINOv2, Street View CLIP, DINOv2 plus SatCLIP, and all representation families without Street View availability metadata. Controls are excluded so that the analysis isolates representation dimension.

In [ ]:
# Define the seven representation sets included in the dimension comparison.
selected_sets = [
    "SatCLIP",
    "TESSERA",
    "AlphaEarth",
    "DINOv2",
    "StreetView_CLIP_only",
    "DINOv2_plus_SatCLIP",
    "All_representations_without_SV_metadata",
]

def dedupe_preserve_order(columns):
    return list(dict.fromkeys(columns))

def columns_sha256(columns):
    return hashlib.sha256(json.dumps(list(columns), separators=(",", ":")).encode("utf-8")).hexdigest()

set_columns = {
    "SatCLIP": list(feature_sets["SatCLIP"]),
    "TESSERA": list(feature_sets["TESSERA"]),
    "AlphaEarth": list(feature_sets["AlphaEarth"]),
    "DINOv2": list(feature_sets["DINOv2"]),
    "StreetView_CLIP_only": list(feature_sets["StreetView_CLIP_only"]),
    "DINOv2_plus_SatCLIP": dedupe_preserve_order(
        list(feature_sets["DINOv2"]) + list(feature_sets["SatCLIP"])
    ),
    "All_representations_without_SV_metadata": list(feature_sets["All_representations_without_SV_metadata"]),
}

expected_native_dims = {
    "SatCLIP": 256,
    "TESSERA": 128,
    "AlphaEarth": 64,
    "DINOv2": 768,
    "StreetView_CLIP_only": 512,
    "DINOv2_plus_SatCLIP": 1024,
    "All_representations_without_SV_metadata": 1728,
}
for name in selected_sets:
    cols = set_columns[name]
    assert len(cols) == expected_native_dims[name]
    assert len(cols) == len(set(cols))
    assert not [c for c in cols if c not in df.columns]
    assert not [c for c in cols if c in categorical_master]
    assert len(cols) >= 64

model_specs = pd.DataFrame([
    {
        "task": task,
        "feature_set": feature_set,
        "model_id": f"{task}__{feature_set}__PCA64",
        "native_dimensions": expected_native_dims[feature_set],
        "pca_dimensions": 64,
        "native_source": "09" if feature_set == "DINOv2_plus_SatCLIP" else "06",
    }
    for task in ["PTAL", "EPC"]
    for feature_set in selected_sets
])
assert len(model_specs) == 14 and model_specs["model_id"].is_unique
display(model_specs)

## 3. Define the 64-component pipeline

Training medians fill missing values, features are standardised, and PCA reduces the representation to 64 components before Ridge fitting. AlphaEarth already has 64 dimensions and acts as a reference case: rotating all 64 dimensions should preserve its performance to practical numerical equivalence. A maximum fold-level R² difference of 0.001 is used as the tolerance for minor solver variation.

In [ ]:
# Build a pipeline that learns imputation, scaling and 64-component PCA from training data.
PCA_CACHE_DIR = Path("/content/notebook10_pca_cache")
PCA_CACHE_DIR.mkdir(parents=True, exist_ok=True)
pca_memory = Memory(location=str(PCA_CACHE_DIR), verbose=0)

def build_pca_pipeline(feature_cols):
    solver = "full" if len(feature_cols) == 64 else "randomized"
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=64, svd_solver=solver, random_state=RANDOM_STATE)),
        ("ridge", Ridge(solver="lsqr", max_iter=5000, tol=1e-4)),
    ], memory=pca_memory)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def atomic_csv(frame, path):
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    frame.to_csv(tmp, index=False)
    tmp.replace(path)

def atomic_parquet(frame, path):
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    frame.to_parquet(tmp, index=False)
    tmp.replace(path)

def atomic_json(obj, path):
    tmp = path.with_name(path.stem + ".tmp" + path.suffix)
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    tmp.replace(path)

def coerce_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series
    mapped = series.astype(str).str.strip().str.lower().map({"true": True, "false": False})
    assert mapped.notna().all()
    return mapped.astype(bool)

## 4. Reuse the same held-out boroughs

The five borough divisions are reconstructed and matched to the main benchmark. This permits direct within-round comparison between each native and compressed representation.

In [ ]:
# Reconstruct the same borough-based test rounds used by the native models.
outer_rows = []
fold_index_by_task = {}
for task in ["PTAL", "EPC"]:
    task_df = df[df["task"] == task].reset_index(drop=True)
    groups = task_df[group_col].astype(str).to_numpy()
    task_fold = np.full(len(task_df), -1, dtype=int)
    for fold, (_, test_idx) in enumerate(GroupKFold(RIDGE_OUTER_SPLITS).split(task_df, task_df[target_col], groups)):
        task_fold[test_idx] = fold
        for idx in test_idx:
            outer_rows.append({
                "sample_id": task_df.loc[idx, "sample_id"], "task": task,
                "outer_fold": int(fold), "borough_code": task_df.loc[idx, group_col],
                "target": float(task_df.loc[idx, target_col]),
            })
    assert (task_fold >= 0).all()
    fold_index_by_task[task] = task_fold

outer_folds = pd.DataFrame(outer_rows).sort_values(["task", "sample_id"], kind="mergesort").reset_index(drop=True)
saved_outer = pd.read_csv(RIDGE_OUTER_FOLDS_PATH).sort_values(["task", "sample_id"], kind="mergesort").reset_index(drop=True)
pd.testing.assert_frame_equal(
    saved_outer[outer_folds.columns], outer_folds,
    check_dtype=False, check_exact=False, rtol=0, atol=1e-12,
)
fold_hash = hashlib.sha256(pd.util.hash_pandas_object(outer_folds, index=False).values.tobytes()).hexdigest()
assert fold_hash == source_06_spec["outer_fold_assignment_sha256"] == source_09_spec["outer_fold_assignment_sha256"]

model_payload = []
for row in model_specs.itertuples(index=False):
    model_payload.append({
        "task": row.task, "feature_set": row.feature_set, "model_id": row.model_id,
        "native_dimensions": int(row.native_dimensions), "pca_dimensions": 64,
        "native_source": row.native_source,
        "feature_columns_sha256": columns_sha256(set_columns[row.feature_set]),
    })

run_spec = {
    "run_spec_version": "10-v1-2026-08-23",
    "notebook": "10_pca64_representation_dimension_sensitivity.ipynb",
    "question": "Does the native-dimension representation ranking persist when every selected representation set is reduced to 64 fold-local principal components?",
    "model_key_sha256": model_key_hash,
    "feature_manifest_sha256": manifest_hash,
    "outer_fold_assignment_sha256": fold_hash,
    "source_06_run_spec_sha256": source_06_spec_sha256,
    "source_09_run_spec_sha256": source_09_spec_sha256,
    "outer_splits": int(RIDGE_OUTER_SPLITS), "inner_splits": int(RIDGE_INNER_SPLITS),
    "alpha_grid": [float(x) for x in RIDGE_ALPHA_GRID],
    "inner_selection_metric": "RMSE",
    "pca": {"n_components": 64, "fit_scope": "inside every inner and outer training fold", "input_scaling": "training-fold StandardScaler", "random_state": int(RANDOM_STATE)},
    "model_specifications": model_payload,
    "inference": "descriptive native-vs-PCA64 fold-paired deltas and rank stability; no fold-level p-values",
}
run_spec_sha256 = hashlib.sha256(json.dumps(run_spec, sort_keys=True).encode("utf-8")).hexdigest()

if PCA64_RUN_SPEC_PATH.exists():
    with open(PCA64_RUN_SPEC_PATH, "r") as f:
        assert json.load(f) == run_spec, "Existing Notebook-10 checkpoints belong to another run specification"
else:
    atomic_json(run_spec, PCA64_RUN_SPEC_PATH)

expected_keys = {
    (row.task, row.feature_set, fold)
    for row in model_specs.itertuples(index=False)
    for fold in range(RIDGE_OUTER_SPLITS)
}
expected_prediction_rows = sum(task_counts[row.task] for row in model_specs.itertuples(index=False))
assert len(expected_keys) == 70
assert expected_prediction_rows == 186179

print("Fold SHA256:", fold_hash)
print("Run-spec SHA256:", run_spec_sha256)
print("Expected runs:", len(expected_keys), "Expected predictions:", expected_prediction_rows)

## 5. Fit the compressed models

Each task and representation is reduced and fitted separately within all five borough rounds. A local cache avoids recomputing the same training transformation for every Ridge penalty, while each completed round retains its own predictions.

In [ ]:
# Fit every compressed representation and save round-level predictions.
if PCA64_RESULTS_PATH.exists():
    completed = pd.read_csv(PCA64_RESULTS_PATH)
    assert {"task", "feature_set", "outer_fold", "run_spec_sha256"}.issubset(completed.columns)
    assert not completed.duplicated(["task", "feature_set", "outer_fold"]).any()
    assert completed["run_spec_sha256"].eq(run_spec_sha256).all()
    completed["outer_fold"] = completed["outer_fold"].astype(int)
else:
    completed = pd.DataFrame()
result_rows = [] if completed.empty else completed.to_dict("records")

def checkpoint_is_valid(path, task, feature_set, fold, expected_ids):
    if not path.exists():
        return False
    try:
        p = pd.read_parquet(path)
        required = {"sample_id", "task", "feature_set", "outer_fold", "borough_code", "y_true", "y_pred", "run_spec_sha256"}
        return (
            required.issubset(p.columns)
            and p["sample_id"].is_unique
            and p["task"].eq(task).all()
            and p["feature_set"].eq(feature_set).all()
            and p["outer_fold"].astype(int).eq(fold).all()
            and p["run_spec_sha256"].eq(run_spec_sha256).all()
            and np.isfinite(p["y_true"]).all() and np.isfinite(p["y_pred"]).all()
            and set(p["sample_id"].astype(str)) == set(pd.Series(expected_ids).astype(str))
        )
    except Exception:
        return False

for task in ["PTAL", "EPC"]:
    task_df = df[df["task"] == task].reset_index(drop=True)
    y = pd.to_numeric(task_df[target_col], errors="raise").to_numpy()
    groups = task_df[group_col].astype(str).to_numpy()
    task_fold = fold_index_by_task[task]

    for feature_set in selected_sets:
        cols = set_columns[feature_set]
        X = task_df[cols]
        for outer_fold in range(RIDGE_OUTER_SPLITS):
            test_idx = np.flatnonzero(task_fold == outer_fold)
            train_idx = np.flatnonzero(task_fold != outer_fold)
            run_key = (task, feature_set, outer_fold)
            pred_file = PCA64_CHUNK_DIR / f"{task}__{feature_set}__PCA64__fold{outer_fold}.parquet"
            current_keys = {(r["task"], r["feature_set"], int(r["outer_fold"])) for r in result_rows}
            if run_key in current_keys and checkpoint_is_valid(
                pred_file, task, feature_set, outer_fold, task_df.iloc[test_idx]["sample_id"]
            ):
                print("SKIP validated checkpoint:", run_key)
                continue

            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            g_train = groups[train_idx]
            assert set(g_train).isdisjoint(set(groups[test_idx]))
            inner_splits = list(GroupKFold(RIDGE_INNER_SPLITS).split(X_train, y_train, g_train))
            for a, b in inner_splits:
                assert set(g_train[a]).isdisjoint(set(g_train[b]))

            search = GridSearchCV(
                build_pca_pipeline(cols), {"ridge__alpha": RIDGE_ALPHA_GRID},
                scoring="neg_root_mean_squared_error", cv=inner_splits,
                refit=True, n_jobs=1, return_train_score=False, error_score="raise",
            )
            t0 = time.time()
            search.fit(X_train, y_train)
            elapsed = time.time() - t0
            pred = search.predict(X_test)
            assert np.isfinite(pred).all()
            best_alpha = float(search.best_params_["ridge__alpha"])

            row = {
                "task": task, "feature_set": feature_set, "outer_fold": int(outer_fold),
                "native_dimensions": int(len(cols)), "pca_dimensions": 64,
                "n_train": int(len(train_idx)), "n_test": int(len(test_idx)),
                "n_train_boroughs": int(len(np.unique(g_train))),
                "n_test_boroughs": int(len(np.unique(groups[test_idx]))),
                "best_alpha": best_alpha,
                "alpha_grid_edge": bool(best_alpha in {float(min(RIDGE_ALPHA_GRID)), float(max(RIDGE_ALPHA_GRID))}),
                "inner_best_rmse": float(-search.best_score_),
                "r2": float(r2_score(y_test, pred)), "rmse": rmse(y_test, pred),
                "mae": float(mean_absolute_error(y_test, pred)),
                "fit_seconds": float(elapsed), "run_spec_sha256": run_spec_sha256,
            }
            pred_frame = pd.DataFrame({
                "sample_id": task_df.iloc[test_idx]["sample_id"].to_numpy(),
                "task": task, "feature_set": feature_set, "outer_fold": int(outer_fold),
                "borough_code": groups[test_idx], "y_true": y_test, "y_pred": pred,
                "run_spec_sha256": run_spec_sha256,
            })
            atomic_parquet(pred_frame, pred_file)
            result_rows = [r for r in result_rows if (r["task"], r["feature_set"], int(r["outer_fold"])) != run_key]
            result_rows.append(row)
            atomic_csv(pd.DataFrame(result_rows).sort_values(["task", "feature_set", "outer_fold"]), PCA64_RESULTS_PATH)
            print(row)
            del search, pred, pred_frame, X_train, X_test
            gc.collect()

print("Notebook-10 fitting complete or safely checkpointed")

## 6. Assemble the complete compressed prediction set

The analysis requires 70 model rounds and 70 matching prediction files. Every held-out sample, borough and outcome is checked against the established evaluation assignments before the summaries are produced.

In [ ]:
# Verify that all compressed predictions match the intended held-out samples.
results = pd.read_csv(PCA64_RESULTS_PATH)
results["outer_fold"] = results["outer_fold"].astype(int)
assert not results.duplicated(["task", "feature_set", "outer_fold"]).any()
assert results["run_spec_sha256"].eq(run_spec_sha256).all()
actual_keys = set(map(tuple, results[["task", "feature_set", "outer_fold"]].to_numpy()))
assert actual_keys == expected_keys, "Notebook 10 incomplete: rerun Section 5"

pred_frames, chunk_rows = [], []
for task, feature_set, fold in sorted(expected_keys):
    path = PCA64_CHUNK_DIR / f"{task}__{feature_set}__PCA64__fold{fold}.parquet"
    assert path.exists()
    p = pd.read_parquet(path)
    required = {"sample_id", "task", "feature_set", "outer_fold", "borough_code", "y_true", "y_pred", "run_spec_sha256"}
    assert required.issubset(p.columns) and p["sample_id"].is_unique
    assert p["task"].eq(task).all() and p["feature_set"].eq(feature_set).all()
    assert p["outer_fold"].astype(int).eq(fold).all() and p["run_spec_sha256"].eq(run_spec_sha256).all()
    expected = outer_folds[(outer_folds["task"] == task) & (outer_folds["outer_fold"] == fold)][["sample_id", "borough_code", "target"]].sort_values("sample_id").reset_index(drop=True)
    observed = p[["sample_id", "borough_code", "y_true"]].sort_values("sample_id").reset_index(drop=True)
    assert expected["sample_id"].equals(observed["sample_id"])
    assert expected["borough_code"].astype(str).equals(observed["borough_code"].astype(str))
    assert np.allclose(expected["target"], observed["y_true"], rtol=0, atol=1e-12)
    pred_frames.append(p)
    chunk_rows.append({"task": task, "feature_set": feature_set, "outer_fold": fold, "n_rows": len(p), "file": path.name})

preds = pd.concat(pred_frames, ignore_index=True)
assert len(preds) == expected_prediction_rows
assert not preds.duplicated(["sample_id", "task", "feature_set", "outer_fold"]).any()
print("Validated runs:", len(actual_keys), "chunks:", len(chunk_rows), "predictions:", len(preds))

## 7. Compare compressed and native performance

Native results come from the main benchmark, except for the DINOv2–SatCLIP pair, which comes from Notebook 09. Differences are calculated within the same borough round; positive R² differences and negative error differences favour the compressed representation.

In [ ]:
# Join compressed and native metrics within the same borough rounds.
native_06 = pd.read_csv(RIDGE_CORE_RESULTS_PATH)
native_09 = pd.read_csv(IMAGE_LOCATION_RESULTS_PATH)
assert len(native_06) == 135 and len(native_09) == 25

paired_rows = []
for spec in model_specs.itertuples(index=False):
    pca_g = results[(results["task"] == spec.task) & (results["feature_set"] == spec.feature_set)][["outer_fold", "r2", "rmse", "mae"]]
    if spec.native_source == "06":
        native_g = native_06[(native_06["task"] == spec.task) & (native_06["feature_set"] == spec.feature_set)][["outer_fold", "r2", "rmse", "mae"]]
        native_id = spec.feature_set
    else:
        native_id = f"{spec.task}_representation_only__DINOv2_plus_SatCLIP"
        native_g = native_09[(native_09["task"] == spec.task) & (native_09["model_id"] == native_id)][["outer_fold", "r2", "rmse", "mae"]]
    paired = pca_g.merge(native_g, on="outer_fold", validate="one_to_one", suffixes=("_pca64", "_native"))
    assert len(paired) == 5
    for row in paired.itertuples(index=False):
        paired_rows.append({
            "task": spec.task, "feature_set": spec.feature_set,
            "native_source": spec.native_source, "native_model_id": native_id,
            "native_dimensions": int(spec.native_dimensions), "pca_dimensions": 64,
            "outer_fold": int(row.outer_fold),
            "pca64_r2": float(row.r2_pca64), "native_r2": float(row.r2_native),
            "delta_r2": float(row.r2_pca64 - row.r2_native),
            "pca64_rmse": float(row.rmse_pca64), "native_rmse": float(row.rmse_native),
            "delta_rmse": float(row.rmse_pca64 - row.rmse_native),
            "pca64_mae": float(row.mae_pca64), "native_mae": float(row.mae_native),
            "delta_mae": float(row.mae_pca64 - row.mae_native),
        })

paired = pd.DataFrame(paired_rows).sort_values(["task", "feature_set", "outer_fold"])
assert len(paired) == 70
display(paired.groupby(["task", "feature_set"])[["delta_r2", "delta_rmse", "delta_mae"]].mean())

## 8. Compare the representation rankings

The summary reports absolute 64-component performance, within-round changes from native dimension and the rank order under both settings. Spearman rank correlation describes how closely the two model orderings agree; it is used as a descriptive summary of seven selected feature sets rather than as a hypothesis test.

In [ ]:
# Summarise performance changes and compare the model rankings.
summary_rows = []
for (task, feature_set), g in results.groupby(["task", "feature_set"]):
    pg = preds[(preds["task"] == task) & (preds["feature_set"] == feature_set)]
    summary_rows.append({
        "task": task, "feature_set": feature_set,
        "native_dimensions": int(g["native_dimensions"].iloc[0]), "pca_dimensions": 64,
        "mean_r2": float(g["r2"].mean()), "sd_r2": float(g["r2"].std(ddof=1)),
        "mean_rmse": float(g["rmse"].mean()), "sd_rmse": float(g["rmse"].std(ddof=1)),
        "mean_mae": float(g["mae"].mean()), "sd_mae": float(g["mae"].std(ddof=1)),
        "pooled_r2": float(r2_score(pg["y_true"], pg["y_pred"])),
        "pooled_rmse": rmse(pg["y_true"], pg["y_pred"]),
        "pooled_mae": float(mean_absolute_error(pg["y_true"], pg["y_pred"])),
        "alpha_edge_hits": int(coerce_bool(g["alpha_grid_edge"]).sum()),
        "total_fit_minutes": float(g["fit_seconds"].sum() / 60),
    })
summary = pd.DataFrame(summary_rows)

comparison_summary = paired.groupby(
    ["task", "feature_set", "native_source", "native_model_id", "native_dimensions", "pca_dimensions"], as_index=False
).agg(
    mean_delta_r2=("delta_r2", "mean"), sd_delta_r2=("delta_r2", "std"),
    r2_wins=("delta_r2", lambda s: int((s > 0).sum())),
    mean_delta_rmse=("delta_rmse", "mean"), rmse_wins=("delta_rmse", lambda s: int((s < 0).sum())),
    mean_delta_mae=("delta_mae", "mean"), mae_wins=("delta_mae", lambda s: int((s < 0).sum())),
)

rank_rows = []
for task in ["PTAL", "EPC"]:
    task_cmp = comparison_summary[comparison_summary["task"] == task].copy()
    native_mean = paired[paired["task"] == task].groupby("feature_set", as_index=False)["native_r2"].mean().rename(columns={"native_r2": "native_mean_r2"})
    pca_mean = summary[summary["task"] == task][["feature_set", "mean_r2"]].rename(columns={"mean_r2": "pca64_mean_r2"})
    ranks = native_mean.merge(pca_mean, on="feature_set", validate="one_to_one")
    ranks["native_rank"] = ranks["native_mean_r2"].rank(ascending=False, method="min").astype(int)
    ranks["pca64_rank"] = ranks["pca64_mean_r2"].rank(ascending=False, method="min").astype(int)
    ranks["task"] = task
    rho = float(ranks["native_mean_r2"].corr(ranks["pca64_mean_r2"], method="spearman"))
    ranks["spearman_rank_correlation_task"] = rho
    rank_rows.append(ranks)
rank_summary = pd.concat(rank_rows, ignore_index=True)[["task", "feature_set", "native_mean_r2", "pca64_mean_r2", "native_rank", "pca64_rank", "spearman_rank_correlation_task"]]

display(summary.sort_values(["task", "mean_r2"], ascending=[True, False]))
display(comparison_summary)
display(rank_summary.sort_values(["task", "pca64_rank"]))

## 9. Finalise the sensitivity results

The complete run matrix, prediction identity and Ridge penalty range are confirmed. The AlphaEarth reference produces a maximum absolute fold-level R² difference of 0.000747, below the 0.001 tolerance, and selects the same penalty values after rotation. This supports interpreting larger changes in the other representations as effects of compression rather than a pipeline inconsistency.

In [ ]:
# Check the AlphaEarth reference rotation and save the sensitivity results.
edge_counts = results.assign(edge=coerce_bool(results["alpha_grid_edge"])).groupby(["task", "feature_set"])["edge"].sum().reset_index(name="edge_hits")
repeated_edge = edge_counts[edge_counts["edge_hits"] >= 3]

alpha_check = comparison_summary[comparison_summary["feature_set"] == "AlphaEarth"]
assert len(alpha_check) == 2
alpha_fold_check = paired[paired["feature_set"] == "AlphaEarth"].copy()
assert len(alpha_fold_check) == 10
alpha_rotation_max_abs_fold_delta_r2 = float(alpha_fold_check["delta_r2"].abs().max())

alpha_pca_alphas = results[results["feature_set"] == "AlphaEarth"][
    ["task", "outer_fold", "best_alpha"]
].rename(columns={"best_alpha": "pca64_best_alpha"})
alpha_native_alphas = native_06[native_06["feature_set"] == "AlphaEarth"][
    ["task", "outer_fold", "best_alpha"]
].rename(columns={"best_alpha": "native_best_alpha"})
alpha_alpha_check = alpha_pca_alphas.merge(
    alpha_native_alphas, on=["task", "outer_fold"],
    how="inner", validate="one_to_one",
)
assert len(alpha_alpha_check) == 10
alphaearth_best_alpha_match_all_folds = bool(np.allclose(
    alpha_alpha_check["pca64_best_alpha"],
    alpha_alpha_check["native_best_alpha"],
    rtol=0, atol=0,
))
alpha_rotation_check_pass = bool(
    alpha_rotation_max_abs_fold_delta_r2 <= 1e-3
    and alphaearth_best_alpha_match_all_folds
)

integrity_gate_pass = bool(
    actual_keys == expected_keys and len(preds) == expected_prediction_rows and len(paired) == 70
    and source_06_audit["integrity_gate_pass"] and source_09_audit["integrity_gate_pass"]
)
interpretation_gate_pass = bool(
    integrity_gate_pass and repeated_edge.empty and alpha_rotation_check_pass
    and source_06_audit["interpretation_gate_pass"] and source_09_audit["interpretation_gate_pass"]
)

audit_summary = {
    "run_spec_path": str(PCA64_RUN_SPEC_PATH), "run_spec_sha256": run_spec_sha256,
    "model_key_sha256": model_key_hash, "feature_manifest_sha256": manifest_hash,
    "outer_fold_assignment_sha256": fold_hash,
    "source_06_run_spec_sha256": source_06_spec_sha256, "source_09_run_spec_sha256": source_09_spec_sha256,
    "selected_feature_sets": int(len(selected_sets)), "pca64_models": int(len(model_specs)),
    "expected_fold_runs": int(len(expected_keys)), "completed_fold_runs": int(len(actual_keys)),
    "expected_prediction_rows": int(expected_prediction_rows), "actual_prediction_rows": int(len(preds)),
    "prediction_chunks_validated": int(len(chunk_rows)), "paired_rows": int(len(paired)),
    "n_alpha_grid_edge_hits": int(coerce_bool(results["alpha_grid_edge"]).sum()),
    "n_models_with_repeated_edge_hits": int(len(repeated_edge)),
    "alphaearth_rotation_r2_tolerance": 1e-3,
    "alphaearth_rotation_max_abs_fold_delta_r2": alpha_rotation_max_abs_fold_delta_r2,
    "alphaearth_best_alpha_match_all_folds": alphaearth_best_alpha_match_all_folds,
    "alphaearth_rotation_check_pass": alpha_rotation_check_pass,
    "ptal_spearman_rank_correlation": float(rank_summary[rank_summary["task"] == "PTAL"]["spearman_rank_correlation_task"].iloc[0]),
    "epc_spearman_rank_correlation": float(rank_summary[rank_summary["task"] == "EPC"]["spearman_rank_correlation_task"].iloc[0]),
    "inference": "descriptive native-vs-PCA64 paired deltas and rank stability; no fold-level p-values",
    "integrity_gate_pass": integrity_gate_pass, "interpretation_gate_pass": interpretation_gate_pass,
}
assert integrity_gate_pass, "Notebook-10 integrity gate failed"
assert interpretation_gate_pass, "Interpretation paused: inspect alpha boundary or AlphaEarth rotation check"

atomic_csv(results.sort_values(["task", "feature_set", "outer_fold"]), PCA64_RESULTS_PATH)
atomic_parquet(preds.sort_values(["task", "feature_set", "sample_id"]), PCA64_PREDICTIONS_PATH)
atomic_csv(summary.sort_values(["task", "mean_r2"], ascending=[True, False]), PCA64_SUMMARY_PATH)
atomic_csv(paired, PCA64_DELTAS_PATH)
atomic_csv(comparison_summary, PCA64_COMPARISON_SUMMARY_PATH)
atomic_csv(rank_summary, PCA64_RANK_SUMMARY_PATH)
atomic_json(audit_summary, PCA64_AUDIT_PATH)

print(json.dumps(audit_summary, indent=2))
print("Notebook 10: INTEGRITY PASS + INTERPRETATION PASS")

## Interpretation

High-dimensional representations benefit from information spread beyond their first 64 principal components, so native dimensions should remain the basis of the main performance table. At the same time, the near-identical ordering under equal dimension shows that DINOv2's relative strength and the advantage of the full fusion are not simply consequences of having more input columns.